# Data Leakage in Machine Learning

Data leakage happens when a model is trained with information that would not be available at prediction time. It makes validation look excellent while hiding real-world failure.

This notebook gives a practical mental model, concrete examples, and safe patterns you can use to avoid leakage in tabular ML workflows.

## Core idea

A feature is safe only if it exists at the exact moment the prediction is made.

Common leakage patterns include:

- Target leakage: a feature is derived from the answer itself.
- Temporal leakage: a feature contains future information.
- Train-test contamination: preprocessing or aggregation uses the full dataset before the split.

If a feature would be unknown in production, it should not be used for training.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(42)
n_rows = 500

df = pd.DataFrame({
    'tenure_months': rng.integers(1, 72, size=n_rows),
    'monthly_charges': rng.normal(70, 15, size=n_rows).round(2),
    'support_calls_last_90_days': rng.poisson(2, size=n_rows),
})

# A realistic target signal with some noise.
logit = (0.03 * df['support_calls_last_90_days'] - 0.02 * df['tenure_months'] + 0.01 * df['monthly_charges'])
prob = 1 / (1 + np.exp(-logit))
df['churn'] = (rng.random(n_rows) < prob).astype(int)

# Leakage feature: it directly copies the target. This would not exist at prediction time.
df['cancellation_reason_present'] = df['churn']

X_safe = df[['tenure_months', 'monthly_charges', 'support_calls_last_90_days']]
X_leaky = df[['tenure_months', 'monthly_charges', 'support_calls_last_90_days', 'cancellation_reason_present']]
y = df['churn']

X_train_safe, X_test_safe, y_train, y_test = train_test_split(X_safe, y, test_size=0.25, random_state=7, stratify=y)
X_train_leaky, X_test_leaky, _, _ = train_test_split(X_leaky, y, test_size=0.25, random_state=7, stratify=y)

safe_model = LogisticRegression(max_iter=1000)
safe_model.fit(X_train_safe, y_train)
safe_accuracy = accuracy_score(y_test, safe_model.predict(X_test_safe))

leaky_model = LogisticRegression(max_iter=1000)
leaky_model.fit(X_train_leaky, y_train)
leaky_accuracy = accuracy_score(y_test, leaky_model.predict(X_test_leaky))

print({
    'safe_accuracy': round(safe_accuracy, 3),
    'leaky_accuracy': round(leaky_accuracy, 3),
})

## Why the leaked feature is dangerous

The leaked column copies the answer into the feature set. The model looks excellent because it can read the label indirectly. In production, that feature would not exist, so the model would fail.

This is the most common leakage pattern: a feature that is only known after the outcome has already happened.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Correct pattern: split first, then fit preprocessing only on training data.
X = df[['tenure_months', 'monthly_charges', 'support_calls_last_90_days']]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=7, stratify=y)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000)),
])

pipeline.fit(X_train, y_train)
pipeline_accuracy = accuracy_score(y_test, pipeline.predict(X_test))
print({'pipeline_accuracy': round(pipeline_accuracy, 3)})

## Prediction moment test

Before using any feature, ask:

1. Would I know this value at the exact moment the prediction is made?
2. Does this value depend on the outcome I am trying to predict?
3. Was this statistic or transformation fit using training data only?

If the answer to any of these is no, treat the feature or step as suspicious.

## Practical checklist

- Split train and test before fitting anything.
- Fit scalers, encoders, imputers, and aggregations on training data only.
- Remove target-derived columns and post-outcome fields.
- Review each feature against the real prediction timestamp.
- Use pipelines so preprocessing stays attached to training data.

## Bonus resources

- Kaggle Tutorial on Data Leakage
- Google Machine Learning Crash Course: Data Preparation